# Baseline probe: 10 positions

Three questions, in order:

1. Does the model produce a move at all, or ramble past the token budget?
2. Can `parse_move` extract it correctly?
3. How good is that move, in centipawns lost against Stockfish?

Runs on Colab (pick a GPU runtime) or locally. The setup cell below installs
Stockfish and the Python deps, clones the repo, and builds a small position
set if `data/positions.json` is missing -- all skipped when it is already
there. Locally it only checks that `stockfish` is on PATH
(`brew install stockfish`).

Generation is slow on CPU/MPS -- a GPU box is the intended home for this.

In [ ]:
# Setup -- installs and clones on Colab, a no-op on a working local checkout.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/AmirBraham/chessllm.git"
REPO_DIR = "/content/chessllm"
N_POSITIONS = 50  # only used when data/positions.json has to be built

try:  # find_spec raises rather than returning None when `google` is absent
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False


def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)


if IN_COLAB:
    sh("apt-get -qq update && apt-get -qq install -y stockfish")
    sh(f"{sys.executable} -m pip install -q "
       f"'chess>=1.11.2' 'transformers>=5.14.1' pyarrow huggingface_hub")

    if not os.path.isdir(REPO_DIR):
        sh(f"git clone -q {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Debian puts the binary in /usr/games, which is not always on PATH here.
    if not shutil.which("stockfish") and os.path.exists("/usr/games/stockfish"):
        os.environ["PATH"] += os.pathsep + "/usr/games"

assert shutil.which("stockfish"), (
    "stockfish not on PATH -- `brew install stockfish` (mac) "
    "or `apt-get install -y stockfish` (linux)"
)

# The dataset is not in the repo; rebuilding it costs a few minutes of
# Stockfish time at depth 12.
if not os.path.exists("data/positions.json"):
    print(f"building data/positions.json ({N_POSITIONS} positions)...")
    sh(f"{sys.executable} build_data.py --n {N_POSITIONS}")

print(f"colab={IN_COLAB}  cwd={os.getcwd()}  stockfish={shutil.which('stockfish')}")

In [ ]:
import json
import statistics as st

import chess

from board import ascii_board, build_prompt, parse_move
from engine import Engine
from qwen3 import generate, get_device, load

N = 10
MAX_NEW_TOKENS = 512
THINK = True  # set False to disable Qwen3 thinking mode

records = json.load(open("data/positions.json"))[:N]
boards = [chess.Board(r["fen"]) for r in records]
print(f"{len(boards)} positions, device = {get_device()}")

## What the model actually sees

In [ ]:
print(build_prompt(boards[0]))

## Load the model

First run downloads ~1.2 GB from Hugging Face.

In [ ]:
model, tok = load()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params on {model.device}")

## Generate

Watch the truncation count. Anything above zero means completions are hitting
the budget before committing to a move, and every move parsed out of those was
scraped from mid-reasoning rather than chosen.

In [ ]:
prompts = [build_prompt(b) for b in boards]
texts, lengths = generate(
    model, tok, prompts, max_new_tokens=MAX_NEW_TOKENS, think=THINK
)

truncated = sum(n >= MAX_NEW_TOKENS for n in lengths)
print(f"mean {st.mean(lengths):.0f} tokens (max {max(lengths)})")
print(f"truncated: {truncated}/{len(lengths)}")
print(f"closed </think>: {sum('</think>' in t for t in texts)}/{len(texts)}")

## One completion in full

The interesting part is whether its reasoning describes the actual position.

In [ ]:
print(texts[0])
print("\n" + "=" * 60)

move, raw = parse_move(boards[0], texts[0])
print(f"raw token : {raw!r}")
print(f"parsed    : {boards[0].san(move) if move else None}")
print(f"legal     : {move is not None}")

## Score all 10

`raw` is the token `parse_move` found; `move` is it validated against the
position. `raw` set with `move` empty means the model named an illegal move --
a different failure from naming none at all.

In [ ]:
with Engine(depth=12, threads=1) as eng:
    rows = []
    for b, text, n_tok in zip(boards, texts, lengths):
        move, raw = parse_move(b, text)
        rows.append(
            {
                "raw": raw,
                "move": b.san(move) if move else None,
                "cp_loss": eng.cp_loss(b, move) if move else None,
                "best": b.san(eng.best(b)[0]),
                "tokens": n_tok,
                "trunc": n_tok >= MAX_NEW_TOKENS,
            }
        )

print(f"{'#':>2} {'raw':>12} {'move':>7} {'cp_loss':>8} {'best':>7} {'tok':>5}  trunc")
for i, r in enumerate(rows):
    cp = "-" if r["cp_loss"] is None else str(r["cp_loss"])
    print(
        f"{i:>2} {str(r['raw'])[:12]:>12} {str(r['move']):>7} {cp:>8} "
        f"{r['best']:>7} {r['tokens']:>5}  {r['trunc']}"
    )

## Against the floor and the ceiling

A cp_loss number means nothing on its own. Random is the floor to beat;
Stockfish's own move is the ceiling, and lands near 10-20 rather than 0 because
the search runs one ply deeper after the move than at the root.

In [ ]:
from baseline import run_random, run_stockfish, summarize

with Engine(depth=12, threads=1) as eng:
    summarize("random legal move", run_random(eng, boards))
    summarize("stockfish best", run_stockfish(eng, boards))

losses = [r["cp_loss"] for r in rows if r["cp_loss"] is not None]
print(f"\nqwen3-0.6B  ({len(losses)} legal of {len(rows)})")
if losses:
    print(f"  mean cp_loss     {st.mean(losses):6.0f}")
    print(f"  median cp_loss   {st.median(losses):6.0f}")
    print(f"  good moves <=50  {sum(x <= 50 for x in losses) / len(losses):6.1%}")

## Where to go from here

- **High truncation** -- raise `MAX_NEW_TOKENS`, or set `THINK = False` and rerun.
  No-think should collapse completions to a few dozen tokens.
- **Low legal rate** -- the legal-move list is in the prompt, so this means the
  model is not reading it. Try `build_prompt(b, include_board=False)` to see
  whether the ASCII board is distracting rather than helping.
- **cp_loss near the random floor** -- expected for the untrained baseline. That
  gap is the room GRPO has to work with.